# Laboratório — Momentum e Nesterov do zero

Este laboratório implementa velocidade, heavy-ball e Nesterov em **NumPy puro**. Os experimentos auditam memória exponencial, estabilidade quadrática, ravinas, ruído e checkpoints.

**Dependências mínimas:** Python 3.11, NumPy 1.26 e Matplotlib 3.8.  
**Seed canônica:** `20260919`.  
**Dados:** funções quadráticas sintéticas, sem download e sem dados pessoais.

Não usamos PyTorch, TensorFlow, JAX, autograd nem otimizadores prontos.

## 1. Ambiente e contratos básicos

A execução falhará cedo se versões, shapes ou valores não respeitarem os contratos. Os gráficos possuem descrição textual na célula que os antecede.

In [ ]:
import platform
import warnings

import matplotlib
import matplotlib.pyplot as plt
import numpy as np

SEED = 20260919
np.set_printoptions(precision=6, suppress=True)

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__}")
print(f"Matplotlib: {matplotlib.__version__}")
print(f"Seed: {SEED}")

assert tuple(map(int, np.__version__.split('.')[:2])) >= (1, 26)
assert tuple(map(int, matplotlib.__version__.split('.')[:2])) >= (3, 8)

## 2. Memória exponencial sob gradiente constante

Na convenção da aula, `v` já é a atualização: $v_{t+1}=\mu v_t-\eta g_t$. Para gradiente constante e $v_0=0$,

$$v_{t+1}=-\eta g\frac{1-\mu^{t+1}}{1-\mu}.$$

Vamos confrontar a recorrência com a forma fechada.

In [ ]:
def velocity_sequence_constant(gradient, learning_rate, momentum, steps):
    v = np.zeros_like(np.asarray(gradient, dtype=float))
    sequence = []
    for _ in range(steps):
        v = momentum * v - learning_rate * gradient
        sequence.append(v.copy())
    return np.asarray(sequence)


g = np.array([2.0, -1.0])
eta = 0.1
mu = 0.9
steps = 60
observed = velocity_sequence_constant(g, eta, mu, steps)
t = np.arange(1, steps + 1)[:, None]
closed = -eta * g * (1 - mu**t) / (1 - mu)
limit = -eta * g / (1 - mu)
max_closed_error = float(np.max(np.abs(observed - closed)))

print(f"Erro máximo recorrência × forma fechada: {max_closed_error:.3e}")
print(f"v_60: {observed[-1]} | limite: {limit}")
print(f"Horizonte aproximado 1/(1-mu): {1/(1-mu):.1f} steps")

assert max_closed_error < 1e-14
assert np.linalg.norm(observed[-1] - limit) < 0.0041

## 3. Persistência contra alternância

Usaremos duas sequências escalares de mesma magnitude: gradiente sempre positivo e gradiente com sinal alternado. A comparação isola o efeito temporal; não simula sozinha uma rede.

**Descrição do gráfico:** duas curvas mostram o módulo da velocidade. A sequência persistente cresce rumo ao limite; a alternante permanece muito menor por cancelamento parcial.

In [ ]:
def velocity_from_gradients(gradients, learning_rate, momentum):
    v = 0.0
    values = []
    for grad in gradients:
        v = momentum * v - learning_rate * grad
        values.append(v)
    return np.asarray(values)


n_steps = 40
persistent = np.ones(n_steps)
alternating = (-1.0) ** np.arange(n_steps)
v_persistent = velocity_from_gradients(persistent, 0.1, 0.9)
v_alternating = velocity_from_gradients(alternating, 0.1, 0.9)
ratio = abs(v_persistent[-1]) / abs(v_alternating[-1])

print(f"|v_40| persistente: {abs(v_persistent[-1]):.6f}")
print(f"|v_40| alternante: {abs(v_alternating[-1]):.6f}")
print(f"Razão persistente/alternante: {ratio:.6f}")

assert abs(v_persistent[-1]) > 0.98
assert abs(v_alternating[-1]) < 0.06
assert ratio > 16

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(np.abs(v_persistent), label="gradiente persistente")
ax.plot(np.abs(v_alternating), label="gradiente alternante")
ax.set(xlabel="step", ylabel="módulo da velocidade", title="Memória exponencial e cancelamento")
ax.legend()
ax.grid(alpha=0.25)
plt.show()

## 4. Atualização segura de dicionários de parâmetros

Em uma MLP, cada tensor precisa de um buffer independente, com mesmo nome, shape e dtype. A função abaixo não muta as entradas e rejeita estados incompletos.

In [ ]:
def momentum_step(params, grads, velocity, learning_rate, momentum):
    if learning_rate <= 0:
        raise ValueError("learning_rate deve ser positivo")
    if not 0 <= momentum < 1:
        raise ValueError("momentum deve estar em [0, 1)")
    if set(params) != set(grads) or set(params) != set(velocity):
        raise KeyError("params, grads e velocity devem ter as mesmas chaves")

    new_params, new_velocity = {}, {}
    for name, value in params.items():
        grad = grads[name]
        old_v = velocity[name]
        if value.shape != grad.shape or value.shape != old_v.shape:
            raise ValueError(f"shape incompatível em {name}")
        if value.dtype != old_v.dtype:
            raise TypeError(f"dtype incompatível em {name}")
        if not np.isfinite(grad).all():
            raise FloatingPointError(f"gradiente não finito em {name}")
        v = momentum * old_v - learning_rate * grad
        new_velocity[name] = v
        new_params[name] = value + v
    return new_params, new_velocity


params = {"W": np.array([[1.0, -2.0], [0.5, 3.0]]), "b": np.zeros(2)}
grads = {"W": np.full((2, 2), 0.25), "b": np.array([0.1, -0.2])}
velocity = {name: np.zeros_like(value) for name, value in params.items()}
updated, velocity_1 = momentum_step(params, grads, velocity, 0.1, 0.9)

assert np.allclose(updated["W"], params["W"] - 0.1 * grads["W"])
assert np.allclose(updated["b"], params["b"] - 0.1 * grads["b"])
assert all(updated[k] is not params[k] for k in params)
print("Shapes, dtypes, chaves e primeiro step: aprovados")

### Contraprovas de validação

Os três defeitos abaixo devem ser detectados: shape incompatível, velocidade faltante e gradiente não finito.

In [ ]:
caught = []
cases = [
    ({"W": np.ones((3, 1)), "b": grads["b"]}, velocity, ValueError),
    (grads, {"W": velocity["W"]}, KeyError),
    ({"W": np.full((2, 2), np.nan), "b": grads["b"]}, velocity, FloatingPointError),
]
for bad_grads, bad_velocity, expected in cases:
    try:
        momentum_step(params, bad_grads, bad_velocity, 0.1, 0.9)
    except expected:
        caught.append(expected.__name__)

print("Defeitos detectados:", caught)
assert caught == ["ValueError", "KeyError", "FloatingPointError"]

## 5. O invariante $\mu=0$

Aplicamos a mesma sequência de gradientes a SGD puro e à função de momentum com $\mu=0$. Parâmetros e atualizações devem coincidir até precisão de máquina.

In [ ]:
rng = np.random.default_rng(SEED)
theta_sgd = rng.normal(size=7)
theta_mom = theta_sgd.copy()
v = np.zeros_like(theta_mom)
gradient_stream = rng.normal(size=(50, 7))

for grad in gradient_stream:
    theta_sgd = theta_sgd - 0.03 * grad
    v = 0.0 * v - 0.03 * grad
    theta_mom = theta_mom + v

mu_zero_error = float(np.max(np.abs(theta_sgd - theta_mom)))
print(f"Erro máximo SGD × momentum(mu=0): {mu_zero_error:.3e}")
assert mu_zero_error == 0.0

## 6. Região de estabilidade do heavy-ball escalar

Para $J(\theta)=\lambda\theta^2/2$, as raízes de

$$r^2-(1+\mu-\eta\lambda)r+\mu=0$$

devem estar dentro do círculo unitário. Auditaremos numericamente a fronteira $0<\eta\lambda<2(1+\mu)$.

In [ ]:
def heavy_ball_spectral_radius(eta_lambda, momentum):
    roots = np.roots([1.0, -(1 + momentum - eta_lambda), momentum])
    return float(np.max(np.abs(roots)))


mu = 0.8
boundary = 2 * (1 + mu)
inside = heavy_ball_spectral_radius(boundary - 1e-3, mu)
at_edge = heavy_ball_spectral_radius(boundary, mu)
outside = heavy_ball_spectral_radius(boundary + 1e-3, mu)

print(f"Fronteira eta*lambda: {boundary:.6f}")
print(f"rho dentro: {inside:.9f} | na fronteira: {at_edge:.9f} | fora: {outside:.9f}")

assert inside < 1
assert np.isclose(at_edge, 1.0, atol=1e-12)
assert outside > 1

**Descrição do gráfico:** mapa de calor do raio espectral para pares $(\mu,\eta\lambda)$. A linha branca tracejada é $2(1+\mu)$; abaixo dela e acima de zero, o raio é menor que 1.

In [ ]:
mus = np.linspace(0.0, 0.99, 100)
eta_lambdas = np.linspace(0.01, 4.1, 160)
rho = np.array([[heavy_ball_spectral_radius(a, m) for a in eta_lambdas] for m in mus])

fig, ax = plt.subplots(figsize=(8, 4.5))
image = ax.imshow(
    rho,
    origin="lower",
    aspect="auto",
    extent=[eta_lambdas.min(), eta_lambdas.max(), mus.min(), mus.max()],
    vmin=0,
    vmax=1.25,
    cmap="viridis",
)
ax.plot(2 * (1 + mus), mus, "w--", linewidth=2, label=r"$\eta\lambda=2(1+\mu)$")
ax.set(xlabel=r"$\eta\lambda$", ylabel=r"$\mu$", title="Raio espectral da recorrência heavy-ball")
ax.legend()
fig.colorbar(image, ax=ax, label=r"$\rho$")
plt.show()

## 7. SGD, heavy-ball e Nesterov numa ravina

Considere $J(x)=\tfrac12x^\top Hx$, com $H=\operatorname{diag}(1,100)$. O número de condição é 100. Para tornar a comparação interpretável, todos usam $\eta=0{,}01$; heavy-ball e Nesterov usam $\mu=0{,}9$. Isso é uma **ablação didática**, não uma busca justa de hiperparâmetros.

**Descrição do gráfico:** à esquerda, trajetórias no plano mostram a aproximação ao mínimo na origem. À direita, a loss em escala logarítmica evidencia que a memória acelera a direção de baixa curvatura.

In [ ]:
H = np.diag([1.0, 100.0])
x0 = np.array([4.0, 4.0])


def quadratic_loss(x):
    return float(0.5 * x @ H @ x)


def optimize_quadratic(method, steps=200, learning_rate=0.01, momentum=0.9, noise=None):
    x = x0.copy()
    v = np.zeros_like(x)
    trajectory = [x.copy()]
    losses = [quadratic_loss(x)]
    for step in range(steps):
        point = x + momentum * v if method == "nesterov" else x
        grad = H @ point
        if noise is not None:
            grad = grad + noise[step]
        v = -learning_rate * grad if method == "sgd" else momentum * v - learning_rate * grad
        x = x + v
        trajectory.append(x.copy())
        losses.append(quadratic_loss(x))
    return np.asarray(trajectory), np.asarray(losses)


results = {name: optimize_quadratic(name) for name in ("sgd", "heavy_ball", "nesterov")}
for name, (_, losses) in results.items():
    hits = np.flatnonzero(losses < 1e-6)
    first_hit = int(hits[0]) if len(hits) else None
    print(f"{name:10s} loss_200={losses[-1]:.9e} | primeiro step < 1e-6: {first_hit}")

assert results["sgd"][1][-1] > 0.1
assert results["heavy_ball"][1][-1] < 3e-7
assert results["nesterov"][1][-1] < 1e-9

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for name, (trajectory, losses) in results.items():
    axes[0].plot(trajectory[:, 0], trajectory[:, 1], label=name)
    axes[1].semilogy(losses, label=name)
axes[0].scatter([0], [0], marker="*", s=120, color="black", label="mínimo")
axes[0].set(xlabel="direção lambda=1", ylabel="direção lambda=100", title="Trajetórias")
axes[1].set(xlabel="step", ylabel="loss", title="Convergência")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.legend()
plt.tight_layout()
plt.show()

## 8. Nesterov realmente avalia outro ponto

Com $H$ diagonal, calcularemos os gradientes no ponto atual e no *lookahead*. A igualdade só ocorreria se a velocidade fosse zero ou estivesse em direção de curvatura nula.

In [ ]:
theta = np.array([2.0, -1.0])
velocity = np.array([-0.5, 0.02])
mu = 0.8
lookahead = theta + mu * velocity
grad_heavy_ball = H @ theta
grad_nesterov = H @ lookahead

print("theta:    ", theta)
print("lookahead:", lookahead)
print("grad(theta):    ", grad_heavy_ball)
print("grad(lookahead):", grad_nesterov)

assert np.allclose(lookahead, [1.6, -0.984])
assert not np.allclose(grad_heavy_ball, grad_nesterov)

## 9. Ruído estocástico com sequência controlada

Repetiremos a ravina com ruído gaussiano adicionado ao gradiente. Cada método recebe exatamente a mesma sequência em cada seed. Os hiperparâmetros permanecem os da ablação anterior; portanto, o resultado demonstra esta configuração, não superioridade universal.

In [ ]:
seeds = np.arange(10)
final_losses = {name: [] for name in ("sgd", "heavy_ball", "nesterov")}
for seed in seeds:
    rng = np.random.default_rng(int(seed))
    noise = rng.normal(0.0, 0.2, size=(200, 2))
    for name in final_losses:
        _, losses = optimize_quadratic(name, noise=noise)
        final_losses[name].append(losses[-1])

for name, values in final_losses.items():
    values = np.asarray(values)
    print(f"{name:10s}: {values.mean():.9f} ± {values.std(ddof=1):.9f}")

assert np.mean(final_losses["heavy_ball"]) < 0.004
assert np.mean(final_losses["nesterov"]) < 0.001
assert np.mean(final_losses["sgd"]) > 0.13

## 10. Duas convenções, duas escalas

Compararemos a velocidade sem fator $(1-\mu)$ com uma média móvel de gradientes. Sob gradiente constante, a primeira tende a uma atualização dez vezes maior quando $\mu=0{,}9$ e o mesmo $\eta$ é reutilizado.

In [ ]:
eta, mu, grad = 0.01, 0.9, 2.0
velocity_update = 0.0
ema_gradient = 0.0
for _ in range(100):
    velocity_update = mu * velocity_update - eta * grad
    ema_gradient = mu * ema_gradient + (1 - mu) * grad

ema_update = -eta * ema_gradient
scale_ratio = abs(velocity_update / ema_update)
print(f"Atualização pela velocidade: {velocity_update:.9f}")
print(f"Atualização pela EMA:        {ema_update:.9f}")
print(f"Razão de magnitudes:         {scale_ratio:.6f}")

assert np.isclose(scale_ratio, 10.0, rtol=3e-5)

## 11. Checkpoint: velocidade faz parte do experimento

Executaremos 80 steps determinísticos. Depois repetiremos 35 steps, salvaremos $(\theta,v)$ e retomaremos os 45 restantes. Uma terceira execução recuperará apenas $\theta$ e zerará a velocidade para provar que os pesos não bastam.

In [ ]:
def heavy_ball_segment(theta, velocity, gradients_fn, start, stop, eta=0.01, mu=0.9):
    theta = theta.copy()
    velocity = velocity.copy()
    for step in range(start, stop):
        grad = gradients_fn(theta, step)
        velocity = mu * velocity - eta * grad
        theta = theta + velocity
    return theta, velocity


def deterministic_grad(theta, step):
    perturbation = 0.03 * np.array([np.sin(step), np.cos(step)])
    return H @ theta + perturbation


theta0 = np.array([4.0, 4.0])
v0 = np.zeros(2)
continuous_theta, continuous_v = heavy_ball_segment(theta0, v0, deterministic_grad, 0, 80)
saved_theta, saved_v = heavy_ball_segment(theta0, v0, deterministic_grad, 0, 35)
resumed_theta, resumed_v = heavy_ball_segment(saved_theta, saved_v, deterministic_grad, 35, 80)
weights_only_theta, _ = heavy_ball_segment(saved_theta, np.zeros_like(saved_v), deterministic_grad, 35, 80)

resume_error = float(np.max(np.abs(continuous_theta - resumed_theta)))
weights_only_error = float(np.max(np.abs(continuous_theta - weights_only_theta)))
print(f"Erro retomada completa: {resume_error:.3e}")
print(f"Erro retomando apenas pesos: {weights_only_error:.9f}")

assert resume_error == 0.0
assert np.array_equal(continuous_v, resumed_v)
assert weights_only_error > 1e-3

## 12. Parâmetros heavy-ball analíticos

Para uma quadrática fortemente convexa com espectro em $[m,L]$, os valores clássicos são

$$\eta^*=\frac{4}{(\sqrt L+\sqrt m)^2},\qquad
\mu^*=\left(\frac{\sqrt L-\sqrt m}{\sqrt L+\sqrt m}\right)^2.$$

Eles são um resultado sob hipóteses estritas, não uma receita automática para MLPs.

In [ ]:
m, L = 1.0, 100.0
eta_star = 4 / (np.sqrt(L) + np.sqrt(m)) ** 2
mu_star = ((np.sqrt(L) - np.sqrt(m)) / (np.sqrt(L) + np.sqrt(m))) ** 2
_, loss_star = optimize_quadratic("heavy_ball", steps=100, learning_rate=eta_star, momentum=mu_star)

print(f"eta*: {eta_star:.9f}")
print(f"mu*:  {mu_star:.9f}")
print(f"loss após 100 steps: {loss_star[-1]:.3e}")

assert np.isclose(eta_star, 4 / 121)
assert np.isclose(mu_star, 81 / 121)
assert loss_star[-1] < 1e-9

## 13. Auditoria final

Os contratos cobrem:

- versões e seed;
- forma fechada da memória;
- cancelamento de gradientes alternantes;
- chaves, shapes, dtype e finitude;
- equivalência com SGD em $\mu=0$;
- fronteira espectral;
- trajetórias determinísticas e estocásticas;
- ponto de avaliação de Nesterov;
- diferença de convenções;
- retomada completa e contraprova sem velocidade;
- parâmetros analíticos da quadrática.

In [ ]:
checks = {
    "memoria_fechada": max_closed_error < 1e-14,
    "cancelamento": ratio > 16,
    "defeitos_detectados": len(caught) == 3,
    "mu_zero": mu_zero_error == 0.0,
    "estabilidade_interna": inside < 1,
    "fronteira_marginal": np.isclose(at_edge, 1.0),
    "instabilidade_externa": outside > 1,
    "ravina_heavy_ball": results["heavy_ball"][1][-1] < 3e-7,
    "ravina_nesterov": results["nesterov"][1][-1] < 1e-9,
    "lookahead_distinto": not np.allclose(grad_heavy_ball, grad_nesterov),
    "convencoes_distintas": np.isclose(scale_ratio, 10.0, rtol=3e-5),
    "retomada_exata": resume_error == 0.0,
    "pesos_insuficientes": weights_only_error > 1e-3,
    "parametros_analiticos": loss_star[-1] < 1e-9,
}
assert all(checks.values())
print(f"Auditoria final: {sum(checks.values())}/{len(checks)} grupos aprovados")

## Conclusões

1. A velocidade é uma soma exponencial de gradientes e possui escala dependente de $\eta$ e $\mu$.
2. Na ravina com condição 100, a configuração didática de heavy-ball e Nesterov avançou muito mais na direção plana do que o SGD com o mesmo $\eta$.
3. Nesterov avaliou de fato o gradiente no *lookahead*.
4. Reutilizar o mesmo learning rate entre convenções com e sem $(1-\mu)$ alterou a atualização por fator próximo de 10.
5. Retomar parâmetros e velocidade reproduziu exatamente a trajetória; recuperar apenas os pesos não reproduziu.

Esses resultados validam mecanismos e contratos. Uma alegação de superioridade em uma MLP exigiria busca pela validação, múltiplas seeds e teste reservado.